# Verzahnungs-Pipeline: Komponententest

Dieses Notebook testet jeden Block der RAG-Pipeline isoliert am Beispiel der Domäne **Verzahnungen**.

| Abschnitt | Getestete Komponenten | Evaluationsziel |
|-----------|----------------------|-----------------|
| 1. Embedder laden | `BGEM3Embedder` | Vektorqualität, Dimension |
| 2. PDF-Loader & SemanticChunker | `PDFLoader`, `SemanticChunker` | Chunk-Größen, semantische Kohärenz |
| 3. Metadaten-Extraktor | `OllamaMetadataExtractor` + `gears.yaml` | Feld-Befüllungsrate, LLM-Extraktion |
| 4. Kosinus-Ähnlichkeit | `BGEM3Embedder`, Cosine-Similarity | Ranking-Qualität pro Frage |
| 5. Retriever | `TwoStageRetriever`, `QdrantStore`, `RandomGearGenerator` | Stage-1-Filter + Stage-2-Vektorsuche |
| 6. LLM-Antwortgenerierung | `OllamaAnswerGenerator` | Quellenverweise, Faktentreue, JSON-Log |

---
**Voraussetzungen:**
- Ollama läuft lokal (`http://localhost:11434`)
- Qdrant läuft als Docker-Container (`docker compose up -d`)
- Embedding-Modell BAAI/bge-m3 ist heruntergeladen (erster Start lädt automatisch)

## 0. Setup & Konfiguration

In [1]:
import sys
import math
import json
import datetime
from pathlib import Path
from IPython.display import display, HTML

# Projekt-Root zum Python-Pfad hinzufügen
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ============================================================
# KONFIGURATION
# ============================================================

PDF_PFAD = Path("DIN 867 -  Evolventenverzahnungen an Stirnrädern.pdf")  

OLLAMA_URL   = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2:3b"              

QDRANT_HOST       = "localhost"
QDRANT_PORT       = 6333
QDRANT_COLLECTION = "verzahnung_test"        

EMBEDDER_DEVICE    = "mps"    
CHUNKER_THRESHOLD  = 0.4    
CHUNKER_MIN_TOKENS = 80
CHUNKER_MAX_TOKENS = 512
OVERLAP_SENTENCES  = 1

SIMILARITY_THRESHOLD = 0.50
TOP_K                = 5

# 3 Test-Fragen ohne Nennung konkreter Zahnrad-Parameter –
# der Kontext (Modul, Typ, ...) kommt vom zufälligen CAD-Generator.
TEST_FRAGEN = [
       "Auf was muss ich bei der Produktion dieses Zahnrads achten?",
]
'''
"Welche Norm gilt für diese Verzahnung?", 
"Wie wird die Tragfähigkeit dieser Verzahnung berechnet?",
"Welche Werkstoffe und Wärmebehandlungen eignen sich für dieses Zahnrad?",
"Auf was muss ich bei der Produktion dieses Zahnrads achten?",
'''

# Logs werden hier gespeichert – ein JSON-File pro Test-Lauf
LOG_DIR = Path("logs")

# Schema und Prompt aus dem Projekt-Root (identisch zur Produktiv-Konfiguration)
SCHEMA_PFAD = PROJECT_ROOT / "schemas" / "gears.yaml"
PROMPT_PFAD = PROJECT_ROOT / "prompts" / "answer_system_prompt.txt"

print("Konfiguration geladen:")
print(f"  PDF:       {PDF_PFAD}")
print(f"  Ollama:    {OLLAMA_URL} ({OLLAMA_MODEL})")
print(f"  Qdrant:    {QDRANT_HOST}:{QDRANT_PORT} / {QDRANT_COLLECTION}")
print(f"  Embedder:  {EMBEDDER_DEVICE}  |  Chunker-Threshold: {CHUNKER_THRESHOLD}")
print(f"  Schema:    {SCHEMA_PFAD.exists()}")
print(f"  Prompt:    {PROMPT_PFAD.exists()}")
print(f"  Log-Dir:   {LOG_DIR}")

Konfiguration geladen:
  PDF:       DIN 867 -  Evolventenverzahnungen an Stirnrädern.pdf
  Ollama:    http://localhost:11434 (llama3.2:3b)
  Qdrant:    localhost:6333 / verzahnung_test
  Embedder:  mps  |  Chunker-Threshold: 0.4
  Schema:    True
  Prompt:    True
  Log-Dir:   logs


### Ausgabe-Hilfsfunktionen

Alle HTML-Darstellungsfunktionen sind in `output_helpers.py` ausgelagert.
Farbschema: schwarze Schrift auf neutralen Grautönen, für lesbare Evaluation optimiert.

In [2]:
# Darstellungsfunktionen aus ausgelagerter Datei importieren
from test_verzahnung.output_helpers import (
    zeige_chunk_statistik,
    zeige_chunks_html,
    zeige_satzaehnlichkeit_svg,
    zeige_metadaten_befuellung,
    zeige_metadaten_html,
    zeige_aehnlichkeit_html,
    zeige_retrieval_html,
    zeige_antwort_html,
)

print("Hilfsfunktionen aus output_helpers.py geladen.")

Hilfsfunktionen aus output_helpers.py geladen.


---
## 1. Embedder laden

`BGEM3Embedder` wird **vor** dem Chunker geladen, da der `SemanticChunker`
den Embedder intern für die Satz-Ähnlichkeitsberechnung benötigt.

Evaluation: Vektordimension und Normierung prüfen.

In [3]:
from app.implementations.embedder_bge_m3 import BGEM3Embedder

print("Lade Embedder (BAAI/bge-m3)... Beim ersten Start: 10-15 Sekunden.")
embedder = BGEM3Embedder(
    model_name="BAAI/bge-m3",
    device=EMBEDDER_DEVICE,
    max_length=8192,
    use_sparse=False,
)
print("Embedder geladen.")

# Schnelltest: Vektordimension und L2-Normierung prüfen
import math
probe = embedder.embed(["Testvektor Verzahnung"]).dense_vectors[0]
laenge = math.sqrt(sum(v * v for v in probe))
print(f"  Dimension: {len(probe)}")
print(f"  L2-Norm:   {laenge:.6f}  (sollte ≈ 1.0 sein)")

/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Lade Embedder (BAAI/bge-m3)... Beim ersten Start: 10-15 Sekunden.
Embedder geladen.
  Dimension: 1024
  L2-Norm:   1.000000  (sollte ≈ 1.0 sein)


---
## 2. PDF-Loader & SemanticChunker

Testet `PDFLoader` (PyMuPDF) und `SemanticChunker` isoliert.
Der semantische Chunker setzt Grenzen dort, wo die Kosinus-Ähnlichkeit
zwischen benachbarten Sätzen unter `CHUNKER_THRESHOLD` fällt.

**Evaluation:** Chunk-Anzahl, Größenverteilung und semantische Kohärenz prüfen.
Sind die Chunk-Grenzen inhaltlich sinnvoll gesetzt?

In [4]:
from app.implementations.pdf_loader_pymupdf import PDFLoader

if not PDF_PFAD.exists():
    raise FileNotFoundError(
        f"PDF nicht gefunden: {PDF_PFAD}\n"
        "Bitte PDF_PFAD in der Konfigurationszelle anpassen."
    )

loader = PDFLoader()
dokument = loader.load(PDF_PFAD)

print("PDF geladen:")
print(f"  Datei:  {PDF_PFAD.name}")
print(f"  Hash:   {dokument.doc_hash[:16]}...")
print(f"  Seiten: {len(dokument.pages)}")
print()
print("Erste 3 Seiten (je max. 300 Zeichen):")
for seite in dokument.pages[:3]:
    vorschau = seite.text[:300].replace("\n", " ")
    print(f"  [Seite {seite.page_number}] {vorschau}...")
    print()

PDF geladen:
  Datei:  DIN 867 -  Evolventenverzahnungen an Stirnrädern.pdf
  Hash:   57659670580625bc...
  Seiten: 3

Erste 3 Seiten (je max. 300 Zeichen):
  [Seite 1] DK 621.833.1 DEUTSCHE NORM Februar 1986 Bezugsprofile für Evolventenverzahnungen an Stirnrädern (Zylinderrädern) für den allgenrieinen Maschinenbau und den Schwermaschinenbau DIN 867 Basic rack for involute teeth of cylindrical gears for general engineering and Ersatz für Ausgabe 09.74 heavy engine...

  [Seite 2] Seite 2 DIN 867 4.3 Zahnhöhe h^, Kopfhöhe h^f, Fußhöhe h^\ Kopfspiel cp, gemeinsame Zahnhöhe Die Zahnhöhe h-p des Bezugsprofils wird durch die Profilbe¬ zugslinie unterteilt in die Kopfhöhe /Zap und die Fußhöhe /zfp. Das Kopfspiel Cp ist die Differenz zwischen der Fußhöhe des Bezugsprofils und der K...

  [Seite 3] DIN 867 Seite 3 4.6 Nutzbare Flanken, Fuß-Formhöhe Die geraden Teile der Zahnflanken bilden die nutzbaren Flanken. Bei stetigem Übergang der Geraden in die Fußrundung ist die Fuß-Formhöhe des Bezu

In [5]:
from app.implementations.chunker_semantic import SemanticChunker

# SemanticChunker erhält den bereits geladenen Embedder – er nutzt ihn
# intern für satzweise Ähnlichkeitsberechnung (Grenz-Detektion).
chunker = SemanticChunker(
    embedder=embedder,
    threshold=CHUNKER_THRESHOLD,
    min_chunk_tokens=CHUNKER_MIN_TOKENS,
    max_chunk_tokens=CHUNKER_MAX_TOKENS,
    overlap_sentences=OVERLAP_SENTENCES,
)

chunks = chunker.chunk(dokument)
print(f"Chunking abgeschlossen: {len(chunks)} Chunks (Threshold={CHUNKER_THRESHOLD})")
print()

# Evaluation: Chunk-Statistik und Größenverteilung
zeige_chunk_statistik(chunks)

Chunking abgeschlossen: 4 Chunks (Threshold=0.4)



In [6]:
# Evaluation: Ähnlichkeitsverlauf über alle Sätze des Dokuments
# Grau = Chunk-Bereich | Weiß = exkludiert (< min_chunk_tokens)
# Grüne Punkte >= Threshold, blaue Punkte < Threshold (→ Grenz-Tick unten)
# Seitenangaben oberhalb der oberen Achsenkante, Legende unterhalb.
zeige_satzaehnlichkeit_svg(
    dokument, embedder,
    threshold=CHUNKER_THRESHOLD,
    min_chunk_tokens=CHUNKER_MIN_TOKENS,
    max_chunk_tokens=CHUNKER_MAX_TOKENS,
    overlap_sentences=OVERLAP_SENTENCES,
    hoehe=340,
)

In [7]:
# Vollständige Chunk-Texte (scrollbar) –
# Evaluation: Chunk-Grenzen auf semantische Kohärenz prüfen
zeige_chunks_html(chunks, titel=f"Alle Chunks aus: {PDF_PFAD.name}", hoehe=800)

---
## 3. Metadaten-Extraktor

Testet `OllamaMetadataExtractor` mit `schemas/gears.yaml`.
Für jeden Chunk wird das LLM aufgerufen und die extrahierten Felder ausgegeben.

**Evaluation:** Welche Felder extrahiert das LLM zuverlässig? Welche werden übersehen?
→ `zeige_metadaten_befuellung` zeigt die Befüllungsrate pro Feld auf einen Blick.

> Hinweis: Dieser Schritt dauert je nach Chunk-Anzahl 1–5 Minuten.

In [8]:
from app.implementations.metadata_extractor_ollama import OllamaMetadataExtractor
from app.core.schema import load_schema

schema = load_schema(SCHEMA_PFAD)
extractor = OllamaMetadataExtractor(
    model_name=OLLAMA_MODEL,
    base_url=OLLAMA_URL,
    timeout_s=60,
    max_retries=2,
)

schema_felder = [f.name for f in schema.fields]
print(f"Schema: {schema.domain} ({len(schema.fields)} Felder)")
print(f"Felder:        {schema_felder}")
print(f"Filter-Felder: {[f.name for f in schema.filter_fields]}")

Schema: Verzahnungen (2 Felder)
Felder:        ['verzahnungstyp', 'modul']
Filter-Felder: ['verzahnungstyp', 'modul']


In [9]:
print(f"Extrahiere Metadaten für {len(chunks)} Chunks...")
print("-" * 60)

metadata_liste = []
for i, chunk in enumerate(chunks):
    print(f"  [{i+1:3d}/{len(chunks)}] Seite {chunk.page_number:3d} ...", end=" ", flush=True)
    meta = extractor.extract(chunk, schema)
    metadata_liste.append(meta)
    non_null = {k: v for k, v in meta.items() if v is not None and v not in ("unspecified", "")}
    print(f"→ {list(non_null.keys()) if non_null else '(keine Felder)'}")

print("-" * 60)
print(f"Fertig. {len(metadata_liste)} Metadaten-Dicts erstellt.")

Extrahiere Metadaten für 4 Chunks...
------------------------------------------------------------
  [  1/4] Seite   1 ... → ['verzahnungstyp', 'profilwinkel']
  [  2/4] Seite   2 ... → (keine Felder)
  [  3/4] Seite   3 ... → (keine Felder)
  [  4/4] Seite   3 ... → ['verzahnungstyp', 'modul']
------------------------------------------------------------
Fertig. 4 Metadaten-Dicts erstellt.


In [10]:
# Evaluation: Befüllungsrate pro Schema-Feld
zeige_metadaten_befuellung(metadata_liste, schema_felder)

In [11]:
# Detailansicht: Chunk-Vorschau + extrahiertes JSON je Chunk
zeige_metadaten_html(chunks, metadata_liste, hoehe=700)

---
## 4. Kosinus-Ähnlichkeit

Berechnet Chunk-Vektoren und Frage-Vektoren mit dem bereits geladenen Embedder,
dann Kosinus-Ähnlichkeit für jede der 3 Test-Fragen.

**Evaluation:** Rankt der Embedder semantisch relevante Chunks nach oben?
Liegt der Top-1-Chunk oberhalb des Threshold?

In [12]:
# Chunk-Vektoren (chunk-level, für Qdrant und Ähnlichkeitsberechnung)
# Hinweis: SemanticChunker nutzt den Embedder satzweise intern;
# für Qdrant benötigen wir separate Chunk-Vektoren.
print(f"Bette {len(chunks)} Chunks ein...")
chunk_vektoren = embedder.embed([c.text for c in chunks]).dense_vectors
print(f"Fertig. Dimension: {len(chunk_vektoren[0])}")

# Frage-Vektoren
print(f"Bette {len(TEST_FRAGEN)} Test-Fragen ein...")
fragen_vektoren = embedder.embed(TEST_FRAGEN).dense_vectors

def kosinus_aehnlichkeit(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a)) or 1.0
    nb = math.sqrt(sum(y * y for y in b)) or 1.0
    return dot / (na * nb)

similarity_pro_frage = {}
for frage, fvec in zip(TEST_FRAGEN, fragen_vektoren):
    scores = sorted(
        [(kosinus_aehnlichkeit(fvec, cvec), chunk)
         for chunk, cvec in zip(chunks, chunk_vektoren)],
        key=lambda x: -x[0],
    )
    similarity_pro_frage[frage] = scores

print()
print("Top-1-Treffer pro Frage:")
print("-" * 80)
for frage, scores in similarity_pro_frage.items():
    beste_score, bester_chunk = scores[0]
    oberhalb = sum(1 for s, _ in scores if s >= SIMILARITY_THRESHOLD)
    vorschau = bester_chunk.text[:120].replace("\n", " ")
    print(f"  Frage:  {frage}")
    print(f"  Score:  {beste_score:.4f}  ({oberhalb} Chunks >= {SIMILARITY_THRESHOLD})")
    print(f"  Chunk:  S.{bester_chunk.page_number}: {vorschau}...")
    print()

Bette 4 Chunks ein...
Fertig. Dimension: 1024
Bette 1 Test-Fragen ein...

Top-1-Treffer pro Frage:
--------------------------------------------------------------------------------
  Frage:  Auf was muss ich bei der Produktion dieses Zahnrads achten?
  Score:  0.4993  (0 Chunks >= 0.5)
  Chunk:  S.1: DK 621.833.1 DEUTSCHE NORM Februar 1986 Bezugsprofile für Evolventenverzahnungen an Stirnrädern (Zylinderrädern) für den...



In [13]:
# Evaluation: vollständiges Ranking aller Chunks pro Frage
# Grün >= Threshold, neutral darunter, gestrichelte Threshold-Linie
for frage, scores in similarity_pro_frage.items():
    zeige_aehnlichkeit_html(frage, scores, threshold=SIMILARITY_THRESHOLD, hoehe=450)
    print()

---
## 5. Retriever (Stage 1 + Stage 2 + Qdrant)

Testet den vollständigen zweistufigen Retriever:
- **Stage 1:** Deterministischer Metadaten-Filter aus dem zufällig generierten CAD-Kontext
- **Stage 2:** Semantische Vektorsuche in Qdrant

**Evaluation:** Greift der Stage-1-Filter korrekt? Liefert Stage-2 die thematisch
passenden Chunks? Bei leerem Ergebnis: `stage1_relax_on_empty=True` beobachten.

**Voraussetzung:** `docker compose up -d`

In [14]:
from app.implementations.qdrant_store import QdrantStore

store = QdrantStore(host=QDRANT_HOST, port=QDRANT_PORT, collection_name=QDRANT_COLLECTION)

print(f"Verbunden mit Qdrant: {QDRANT_HOST}:{QDRANT_PORT}")
print(f"Collection: {QDRANT_COLLECTION}")

try:
    collections = store.client.get_collections().collections
    existiert = any(c.name == QDRANT_COLLECTION for c in collections)
    if existiert:
        info = store.client.get_collection(QDRANT_COLLECTION)
        print(f"Collection existiert bereits: {info.points_count} Punkte")
    else:
        print("Collection noch nicht vorhanden – wird beim ersten upsert() angelegt.")
except Exception as e:
    print(f"Verbindungsfehler: {e}")
    print("  Stelle sicher dass Qdrant läuft: docker compose up -d")

Verbunden mit Qdrant: localhost:6333
Collection: verzahnung_test
Collection existiert bereits: 99 Punkte


In [15]:
# OPTIONAL: Collection löschen für sauberen Neustart (z.B. nach Chunker-Änderung)
LOESCHE_COLLECTION = False

if LOESCHE_COLLECTION:
    try:
        store.client.delete_collection(QDRANT_COLLECTION)
        print(f"Collection '{QDRANT_COLLECTION}' gelöscht.")
    except Exception as e:
        print(f"Konnte Collection nicht löschen: {e}")

In [16]:
from app.pipeline.indexer import _sanitize_metadata
from app.core.types import EmbeddedChunk

saubere_metadaten = [_sanitize_metadata(schema, meta) for meta in metadata_liste]

embedded_chunks = [
    EmbeddedChunk(chunk=chunk, dense_vector=vektor, sparse_vector=None, metadata=meta)
    for chunk, vektor, meta in zip(chunks, chunk_vektoren, saubere_metadaten)
]

print(f"Lade {len(embedded_chunks)} Chunks in Qdrant hoch...")
store.upsert(embedded_chunks)
print("Fertig.")

info = store.client.get_collection(QDRANT_COLLECTION)
print(f"Punkte in Collection: {info.points_count}")

Lade 4 Chunks in Qdrant hoch...
Fertig.
Punkte in Collection: 103


In [17]:
from app.implementations.retriever_two_stage import TwoStageRetriever, _build_stage1_filter
from app.implementations.cad_random_gear import RandomGearGenerator

# Zufälligen CAD-Kontext generieren (wie im Produktiv-System)
cad_generator = RandomGearGenerator()
CAD_KONTEXT = cad_generator.extract()

print("Generierter CAD-Kontext (zufällig, nach DIN 3960):")
print(json.dumps(CAD_KONTEXT, ensure_ascii=False, indent=2))
print()

retriever = TwoStageRetriever(
    embedder=embedder,
    store=store,
    schema_path=SCHEMA_PFAD,
    stage1_strict=True,
    stage1_relax_on_empty=True,
    stage1_min_candidates=3,
    top_k=TOP_K,
    min_similarity=SIMILARITY_THRESHOLD,
)

filter_bedingungen = _build_stage1_filter(retriever.schema, CAD_KONTEXT, relax_factor=1.0)
print("Stage-1-Filter:")
print(json.dumps(filter_bedingungen, ensure_ascii=False, indent=2))

Generierter CAD-Kontext (zufällig, nach DIN 3960):
{
  "verzahnungstyp": "Schrägverzahnung",
  "modul": 2.0,
  "zaehnezahl": 48,
  "eingriffswinkel": 20.0,
  "schraegungswinkel": 8.0,
  "profilverschiebung": 0.09,
  "teilkreisdurchmesser": 96.0,
  "kopfkreisdurchmesser": 100.36,
  "fusskreisdurchmesser": 91.36,
  "zahnbreite": 23.4,
  "werkstoff": "C45",
  "haerte": "einsatzgehärtet",
  "verzahnungsqualitaet": 7,
  "drehrichtung": "rechts"
}

Stage-1-Filter:
{
  "must": [
    {
      "key": "metadata.verzahnungstyp",
      "match": "Schrägverzahnung",
      "or_empty": true
    },
    {
      "key": "metadata.modul_min",
      "range": {
        "lte": 2.0
      },
      "or_empty": true
    },
    {
      "key": "metadata.modul_max",
      "range": {
        "gte": 2.0
      },
      "or_empty": true
    }
  ]
}


In [18]:
retrieval_ergebnisse = {}

for frage in TEST_FRAGEN:
    treffer = retriever.retrieve(frage, CAD_KONTEXT)
    retrieval_ergebnisse[frage] = treffer

    # Evaluation: Stage-1-Filter-Bedingungen + Stage-2-Treffer mit Score und Metadaten
    zeige_retrieval_html(frage, treffer, filter_bedingungen=filter_bedingungen, hoehe=550)

    print(f"  {len(treffer)} Treffer")
    for i, rc in enumerate(treffer):
        nm = {k: v for k, v in rc.metadata.items() if v is not None and v not in ("unspecified", "")}
        print(f"    [Q{i+1}] Score={rc.similarity:.3f}  S.{rc.chunk.page_number}  meta={list(nm.keys())}")
    print()

  3 Treffer
    [Q1] Score=0.589  S.14  meta=['verzahnungstyp']
    [Q2] Score=0.535  S.3  meta=['verzahnungstyp']
    [Q3] Score=0.535  S.3  meta=['verzahnungstyp']



---
## 6. LLM-Antwortgenerierung

Testet `OllamaAnswerGenerator` mit `prompts/answer_system_prompt.txt`.

Die extrahierten Metadaten werden im Chunks-Block als **"Extrahierte Fakten"** mitübergeben,
damit das LLM strukturierte Werte (Modul, Norm, ...) direkt zitieren kann.

**Evaluation:**
- Zitiert die Antwort `[Q1]`-Verweise korrekt?
- Stimmen genannte Fakten mit den extrahierten Metadaten überein?
- Ein JSON-Log wird pro Lauf in `logs/` gespeichert für spätere Auswertung.

In [19]:
from app.implementations.answer_generator_ollama import OllamaAnswerGenerator

antwort_gen = OllamaAnswerGenerator(
    model_name=OLLAMA_MODEL,
    base_url=OLLAMA_URL,
    timeout_s=120,
    prompt_path=PROMPT_PFAD,
    domain_name="Verzahnungen",
    max_tokens=1000,
    temperature=0.2,
)

print(f"Antwortgenerator bereit.")
print(f"  Modell:  {OLLAMA_MODEL}")
print(f"  Prompt:  {PROMPT_PFAD.name}")

Antwortgenerator bereit.
  Modell:  llama3.2:3b
  Prompt:  answer_system_prompt.txt


In [20]:
alle_antworten = []
alle_treffer   = []

for frage in TEST_FRAGEN:
    treffer = retrieval_ergebnisse.get(frage, [])
    alle_treffer.append(treffer)

    if not treffer:
        print(f"Keine Treffer → Frage übersprungen: {frage}")
        alle_antworten.append(None)
        continue

    print(f"Generiere Antwort für: {frage!r}  ({len(treffer)} Chunks)...")
    antwort = antwort_gen.generate(
        question=frage,
        chunks=treffer,
        cad_metadata=CAD_KONTEXT,
    )
    alle_antworten.append(antwort)

    # Evaluation: CAD-Kontext, Antworttext mit [Qx]-Hervorhebung, aufklappbare Quellentabelle
    zeige_antwort_html(frage, antwort, cad_kontext=CAD_KONTEXT)
    print()

# ── JSON-Log speichern ─────────────────────────────────────────────────────
LOG_DIR.mkdir(parents=True, exist_ok=True)
zeitstempel = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_pfad = LOG_DIR / f"{zeitstempel}.json"

log_daten = {
    "zeitstempel": zeitstempel,
    "modell": OLLAMA_MODEL,
    "cad_kontext": CAD_KONTEXT,
    "chunker_threshold": CHUNKER_THRESHOLD,
    "similarity_threshold": SIMILARITY_THRESHOLD,
    "fragen": [],
}

for frage, antwort, treffer in zip(TEST_FRAGEN, alle_antworten, alle_treffer):
    eintrag = {
        "frage": frage,
        "treffer_anzahl": len(treffer),
        "treffer": [
            {
                "rank": i + 1,
                "score": round(rc.similarity, 5),
                "seite": rc.chunk.page_number,
                "quelle": Path(rc.chunk.source_path).name,
                "metadaten": {
                    k: v for k, v in rc.metadata.items()
                    if v is not None and v not in ("unspecified", "")
                },
                "text_vorschau": rc.chunk.text[:200],
            }
            for i, rc in enumerate(treffer)
        ],
        "antwort_text": antwort.get("answer_text", "") if antwort else None,
        "quellen": antwort.get("sources", []) if antwort else [],
    }
    log_daten["fragen"].append(eintrag)

with open(log_pfad, "w", encoding="utf-8") as f:
    json.dump(log_daten, f, ensure_ascii=False, indent=2)

print(f"\nLog gespeichert: {log_pfad}")
print(f"  {sum(1 for a in alle_antworten if a)} von {len(TEST_FRAGEN)} Antworten generiert.")

Generiere Antwort für: 'Auf was muss ich bei der Produktion dieses Zahnrads achten?'  (3 Chunks)...


Ref,Datei,Seite,Score
[Q1],verzahnung_wissensbasis_ausfuehrlich.pdf,S.14,0.5894
[Q2],verzahnung_wissensbasis_ausfuehrlich.pdf,S.3,0.5353
[Q3],verzahnung_wissensbasis_ausfuehrlich.pdf,S.3,0.5353




Log gespeichert: logs/20260521_154921.json
  1 von 1 Antworten generiert.


---
## Zusammenfassung

| Abschnitt | Komponente | Evaluation |
|-----------|-----------|------------|
| 1 | `BGEM3Embedder` | Dimension 1024, L2-Norm ≈ 1.0 |
| 2 | `PDFLoader` + `SemanticChunker` | Chunk-Kohärenz, Größenverteilung |
| 3 | `OllamaMetadataExtractor` + `gears.yaml` | Feld-Befüllungsrate, LLM-Extraktion |
| 4 | Kosinus-Ähnlichkeit | Ranking-Qualität, Threshold-Separation |
| 5 | `QdrantStore` + `TwoStageRetriever` + `RandomGearGenerator` | Stage-1-Filter, Stage-2-Vektorsuche |
| 6 | `OllamaAnswerGenerator` | Quellenverweise [Q1], Faktentreue, JSON-Log in `logs/` |